<a href="https://colab.research.google.com/github/codernikita/Tredence_analytics/blob/main/case_study_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import copy
import math
import time
import random
from typing import Dict, List, Tuple

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter
import seaborn as sns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

PyTorch version : 2.10.0+cu128
CUDA available  : True


In [ ]:
SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)


In [ ]:
class PrunableLinear(nn.Module):


    def __init__(self, in_features: int, out_features: int,
                 temperature: float = 0.5) -> None:
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.temperature  = temperature


        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))


        self.gate_scores = nn.Parameter(
            torch.zeros(out_features, in_features))

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        nn.init.normal_(self.gate_scores, mean=0.0, std=0.02)

        fan_in = self.in_features
        bound  = 1.0 / math.sqrt(fan_in)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        gates = torch.sigmoid(self.gate_scores / self.temperature)

        pruned_weights = self.weight * gates

        return F.linear(x, pruned_weights, self.bias)

    @torch.no_grad()
    def get_gates(self) -> torch.Tensor:
        """Detached gate values on CPU. Use for analysis, not training."""
        return torch.sigmoid(self.gate_scores / self.temperature).cpu()

    @torch.no_grad()
    def sparsity(self, threshold: float = 1e-2) -> float:
        """Fraction of gates effectively at zero (below threshold)."""
        return (self.get_gates() < threshold).float().mean().item()

    def extra_repr(self) -> str:
        return (f"in={self.in_features}, out={self.out_features}, "
                f"τ={self.temperature}")


In [ ]:
class PrunableConv2d(nn.Module):


    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, temperature: float = 0.5) -> None:
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding
        self.temperature  = temperature


        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels))

        self.gate_scores = nn.Parameter(
            torch.zeros(out_channels, in_channels, kernel_size, kernel_size))

        self._init_weights()

    def _init_weights(self) -> None:

        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")
        nn.init.normal_(self.gate_scores, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates = torch.sigmoid(self.gate_scores / self.temperature)
        pruned_weights = self.weight * gates
        return F.conv2d(x, pruned_weights, self.bias,
                        self.stride, self.padding)

    @torch.no_grad()
    def get_gates(self) -> torch.Tensor:
        return torch.sigmoid(self.gate_scores / self.temperature).cpu()

    @torch.no_grad()
    def sparsity(self, threshold: float = 1e-2) -> float:
        return (self.get_gates() < threshold).float().mean().item()

    def extra_repr(self) -> str:
        return (f"in={self.in_channels}, out={self.out_channels}, "
                f"k={self.kernel_size}, τ={self.temperature}")

In [ ]:
class PrunableCNN(nn.Module):
    CONV_LAYER_NAMES = ["conv1", "conv2", "conv3", "conv4"]
    FC_LAYER_NAMES   = ["fc1", "fc2"]
    ALL_LAYER_NAMES  = CONV_LAYER_NAMES + FC_LAYER_NAMES

    def __init__(self, num_classes: int = 10, temperature: float = 0.5) -> None:
        super().__init__()
        self.conv1 = PrunableConv2d(3,  32, 3, padding=1, temperature=temperature)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = PrunableConv2d(32, 64, 3, padding=1, temperature=temperature)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = PrunableConv2d(64,  128, 3, padding=1, temperature=temperature)
        self.bn3   = nn.BatchNorm2d(128)
        self.conv4 = PrunableConv2d(128, 128, 3, padding=1, temperature=temperature)
        self.bn4   = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(0.4)
        self.fc1 = PrunableLinear(128 * 8 * 8, 512, temperature=temperature)
        self.fc2 = PrunableLinear(512, num_classes, temperature=temperature)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)



    def _all_prunable(self) -> List[Tuple[str, object]]:
        layers = []
        for n in self.CONV_LAYER_NAMES:
            layers.append((n, getattr(self, n)))
        for n in self.FC_LAYER_NAMES:
            layers.append((n, getattr(self, n)))
        return layers

    @torch.no_grad()
    def overall_sparsity(self, threshold: float = 1e-2) -> float:
        # We call self._all_prunable() because it is a method of this class
        all_gates = torch.cat([l.get_gates().view(-1)
                               for _, l in self._all_prunable()])
        return (all_gates < threshold).float().mean().item()

    @torch.no_grad()
    def layerwise_sparsity(self, threshold: float = 1e-2) -> Dict[str, float]:
        return {n: l.sparsity(threshold) for n, l in self._all_prunable()}

    @torch.no_grad()
    def all_gate_values(self) -> torch.Tensor:
        return torch.cat([l.get_gates().view(-1)
                          for _, l in self._all_prunable()])

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def num_active_weights(self, threshold: float = 1e-2) -> int:
        total_active = 0
        for _, layer in self._all_prunable():
            gates = layer.get_gates()
            total_active += (gates >= threshold).sum().item()
        return total_active

In [ ]:
def sparsity_loss(model: PrunableCNN) -> torch.Tensor:

    device = next(model.parameters()).device
    total  = torch.zeros(1, device=device)
    for _, layer in model._all_prunable():
        # sigmoid(gate_scores / τ) is differentiable w.r.t. gate_scores
        total = total + torch.sigmoid(
            layer.gate_scores / layer.temperature).sum()
    return total.squeeze()


def total_loss(logits: torch.Tensor, targets: torch.Tensor,
               model: PrunableCNN, lam: float,
               criterion: nn.Module) -> Tuple[torch.Tensor, ...]:

    cls = criterion(logits, targets)
    sp  = sparsity_loss(model)
    return cls + lam * sp, cls, sp


def lambda_schedule(base_lam: float, epoch: int,
                    total_epochs: int, warmup: int = 5) -> float:

    if epoch <= warmup:
        return base_lam * 0.1
    progress = (epoch - warmup) / max(total_epochs - warmup, 1)
    return base_lam * min(progress, 1.0)


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
CLASSES = ["airplane","automobile","bird","cat","deer",
           "dog","frog","horse","ship","truck"]

def get_cifar10_loaders(data_dir: str = "./data",
                        batch_size: int = 128,
                        num_workers: int = 2
                        ) -> Tuple[DataLoader, DataLoader]:

    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])

    train_ds = torchvision.datasets.CIFAR10(
        root=data_dir, train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10(
        root=data_dir, train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    print(f"Train samples: {len(train_ds):,}  |  Test samples: {len(test_ds):,}")
    print(f"Batch size: {batch_size}  |  Train batches: {len(train_loader)}")
    return train_loader, test_loader


In [ ]:
def train_one_epoch(model: PrunableCNN, loader: DataLoader,
                    optimizer: optim.Optimizer, criterion: nn.Module,
                    lam_eff: float, device: torch.device
                    ) -> Tuple[float, float, float]:

    model.train()
    t_sum = c_sum = s_sum = 0.0
    n = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        logits = model(images)
        t_loss, c_loss, s_loss = total_loss(logits, labels, model,
                                            lam_eff, criterion)
        t_loss.backward()
        # Clip gradients — gate_scores can blow up early in training
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        t_sum += t_loss.item(); c_sum += c_loss.item(); s_sum += s_loss.item()
        n += 1

    return t_sum / n, c_sum / n, s_sum / n


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader,
             device: torch.device) -> Tuple[float, np.ndarray]:

    model.eval()
    class_correct = np.zeros(10)
    class_total   = np.zeros(10)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        for c in range(10):
            mask = (labels == c)
            class_correct[c] += (preds[mask] == labels[mask]).sum().item()
            class_total[c]   += mask.sum().item()
    per_class_acc = 100.0 * class_correct / (class_total + 1e-8)
    overall_acc   = 100.0 * class_correct.sum() / class_total.sum()
    return overall_acc, per_class_acc


In [ ]:
class BaselineCNN(nn.Module):


    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32,  64,  3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1   = nn.Linear(128 * 8 * 8, 512)
        self.fc2   = nn.Linear(512, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def train_baseline(train_loader: DataLoader, test_loader: DataLoader,
                   device: torch.device, num_epochs: int = 30) -> dict:

    set_seed(SEED)
    print("\n" + "─" * 60)
    print("  Training BASELINE (no pruning) for fair comparison")
    print("─" * 60)

    model     = BaselineCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = []
    best_acc = -1.0

    for epoch in range(1, num_epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        acc, _ = evaluate(model, test_loader, device)
        history.append({"epoch": epoch, "test_acc": acc})
        best_acc = max(best_acc, acc)
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>2}/{num_epochs}  |  Test acc: {acc:.2f}%")

    print(f"\n  ▶ Baseline Best Accuracy: {best_acc:.2f}%\n")
    return {"best_acc": best_acc, "history": history, "model": model}

In [ ]:
def hard_prune(model: PrunableCNN, threshold: float = 1e-2) -> PrunableCNN:

    pruned = copy.deepcopy(model)
    pruned.eval()
    with torch.no_grad():
        for _, layer in pruned._all_prunable():
            gates = torch.sigmoid(layer.gate_scores / layer.temperature)
            mask  = (gates >= threshold).float()
            layer.weight.mul_(mask)
    return pruned

In [ ]:
COL = {
    "low":    "#1B4F72",
    "mid":    "#C0392B",
    "high":   "#1E8449",
    "base":   "#7D3C98",
    "accent": "#F39C12",
    "light":  "#ECF0F1",
    "grid":   "#E5E8E8",
}

In [ ]:
def _ax_style(ax, title="", xlabel="", ylabel="") -> None:

    ax.set_facecolor("#F9FAFB")
    ax.grid(color=COL["grid"], linewidth=0.7, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:  ax.set_title(title,   fontsize=11, fontweight="bold", pad=8)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

In [ ]:
def plot1_gate_distributions(results: dict, out: str) -> None:

    lams   = sorted(results.keys())
    cols   = [COL["low"], COL["mid"], COL["high"]]
    fig, axes = plt.subplots(1, len(lams), figsize=(5.5 * len(lams), 4.5),
                             sharey=False)
    fig.suptitle("Plot 1 — Final Gate Value Distributions",
                 fontsize=14, fontweight="bold", y=1.03)
    if len(lams) == 1: axes = [axes]

    for ax, lam, col in zip(axes, lams, cols):
        gates = results[lam]["gate_values"].numpy()
        acc   = results[lam]["acc_after"]
        spar  = results[lam]["sparsity"] * 100

        n_pruned = (gates < 0.01).sum()
        n_total  = len(gates)

        ax.hist(gates, bins=120, color=col, edgecolor="none",
                alpha=0.80, density=True, zorder=3)
        ax.axvline(0.01, color=COL["accent"], linestyle="--",
                   linewidth=1.8, label="threshold = 0.01", zorder=5)
        ax.text(0.5, 0.92, f"{n_pruned:,}/{n_total:,} pruned",
                transform=ax.transAxes, ha="center",
                fontsize=9, color=col, fontweight="bold")
        _ax_style(ax,
                  f"λ = {lam:.0e}   |   Acc = {acc:.1f}%   |   Sparsity = {spar:.1f}%",
                  "Gate value", "Density")
        ax.legend(fontsize=8, framealpha=0.7)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot2_training_curves(results: dict, baseline: dict, out: str) -> None:

    lams = sorted(results.keys())
    cols = [COL["low"], COL["mid"], COL["high"]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Plot 2 — Training Dynamics", fontsize=14,
                 fontweight="bold", y=1.02)

    # Baseline on accuracy plot
    ep_base = [r["epoch"]    for r in baseline["history"]]
    ac_base = [r["test_acc"] for r in baseline["history"]]
    ax1.plot(ep_base, ac_base, color=COL["base"], linewidth=2.5,
             linestyle=":", label=f"Baseline (no pruning) — {baseline['best_acc']:.1f}%",
             zorder=5)

    for lam, col in zip(lams, cols):
        h    = results[lam]["history"]
        ep   = [r["epoch"]        for r in h]
        acc  = [r["test_acc"]     for r in h]
        spar = [r["sparsity"] * 100 for r in h]
        ax1.plot(ep, acc,  color=col, linewidth=2.2, label=f"λ={lam:.0e}")
        ax2.plot(ep, spar, color=col, linewidth=2.2, label=f"λ={lam:.0e}")

    _ax_style(ax1, "Test Accuracy vs Epoch", "Epoch", "Accuracy (%)")
    _ax_style(ax2, "Network Sparsity vs Epoch", "Epoch", "Sparsity (%)")
    ax1.legend(fontsize=9, framealpha=0.85)
    ax2.legend(fontsize=9, framealpha=0.85)

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot3_lambda_schedule(base_lambdas: List[float],
                          total_epochs: int, out: str) -> None:

    epochs = list(range(1, total_epochs + 1))
    fig, ax = plt.subplots(figsize=(9, 4))
    fig.suptitle("Plot 3 — Lambda Warm-up Schedule",
                 fontsize=14, fontweight="bold")

    for lam, col in zip(sorted(base_lambdas), [COL["low"], COL["mid"], COL["high"]]):
        sched = [lambda_schedule(lam, e, total_epochs) for e in epochs]
        ax.plot(epochs, sched, color=col, linewidth=2.5,
                label=f"λ_base = {lam:.0e}")

    ax.axvspan(1, 5, alpha=0.07, color="gray", label="Warm-up zone (ep 1–5)")
    ax.axvline(5, color="gray", linestyle=":", linewidth=1.5)
    _ax_style(ax, xlabel="Epoch", ylabel="Effective λ (sparsity pressure)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot4_layerwise_sparsity(results: dict, out: str) -> None:

    lams   = sorted(results.keys())
    cols   = [COL["low"], COL["mid"], COL["high"]]
    layers = PrunableCNN.ALL_LAYER_NAMES
    x      = np.arange(len(layers))
    w      = 0.25

    fig, ax = plt.subplots(figsize=(11, 5))
    fig.suptitle("Plot 4 — Layer-wise Sparsity After Training",
                 fontsize=14, fontweight="bold")

    for i, (lam, col) in enumerate(zip(lams, cols)):
        sp   = results[lam]["final_lw_sparsity"]
        vals = [sp.get(n, 0) * 100 for n in layers]
        bars = ax.bar(x + i * w, vals, w, label=f"λ={lam:.0e}",
                      color=col, alpha=0.85, edgecolor="white", zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{v:.0f}%", ha="center", va="bottom",
                    fontsize=7.5, fontweight="bold", color=col)

    ax.set_xticks(x + w)
    ax.set_xticklabels(layers, fontsize=10)
    ax.set_ylim(0, 108)
    # Vertical separator between conv and FC sections
    ax.axvline(3.65, color="gray", linewidth=1.2, linestyle="--", alpha=0.6)
    ax.text(1.5, 103, "← Conv layers →", ha="center", fontsize=9,
            color="gray", style="italic")
    ax.text(4.5, 103, "← FC layers →", ha="center", fontsize=9,
            color="gray", style="italic")
    _ax_style(ax, ylabel="Sparsity (%)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot5_tradeoff(results: dict, baseline_acc: float, out: str) -> None:

    lams  = sorted(results.keys())
    cols  = [COL["low"], COL["mid"], COL["high"]]
    spars = [results[l]["sparsity"] * 100  for l in lams]
    accs  = [results[l]["acc_after"]       for l in lams]

    fig, ax = plt.subplots(figsize=(8, 5.5))
    fig.suptitle("Plot 5 — Accuracy vs Sparsity Trade-off",
                 fontsize=14, fontweight="bold")

    # Baseline reference line
    ax.axhline(baseline_acc, color=COL["base"], linestyle=":",
               linewidth=2, label=f"Baseline (0% sparse) — {baseline_acc:.1f}%")

    # Connect points
    ax.plot(spars, accs, color="#555", linewidth=1.5, linestyle="--", zorder=2)

    for spar, acc, lam, col in zip(spars, accs, lams, cols):
        ax.scatter(spar, acc, s=250, color=col, zorder=5,
                   edgecolors="white", linewidths=1.8)
        ax.annotate(f"λ={lam:.0e}\n{acc:.1f}% acc\n{spar:.0f}% sparse",
                    xy=(spar, acc), xytext=(12, 8),
                    textcoords="offset points", fontsize=8.5,
                    color=col, fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color=col, lw=0.9))

    _ax_style(ax, xlabel="Sparsity (%)", ylabel="Test Accuracy (%)")
    ax.xaxis.set_major_formatter(PercentFormatter())
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot6_before_after(results: dict, out: str) -> None:

    lams   = sorted(results.keys())
    cols   = [COL["low"], COL["mid"], COL["high"]]
    x      = np.arange(len(lams))
    w      = 0.35
    before = [results[l]["acc_before"] for l in lams]
    after  = [results[l]["acc_after"]  for l in lams]
    labels = [f"λ={l:.0e}" for l in lams]

    fig, ax = plt.subplots(figsize=(9, 5))
    fig.suptitle("Plot 6 — Accuracy Before vs After Hard Pruning",
                 fontsize=14, fontweight="bold")

    b1 = ax.bar(x - w/2, before, w, label="Before hard pruning",
                color=COL["low"], alpha=0.85, edgecolor="white", zorder=3)
    b2 = ax.bar(x + w/2, after,  w, label="After hard pruning",
                color=COL["mid"], alpha=0.85, edgecolor="white", zorder=3)

    for bars, vals in [(b1, before), (b2, after)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.4,
                    f"{v:.1f}%", ha="center", va="bottom",
                    fontsize=8.5, fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, max(before + after) + 12)
    _ax_style(ax, ylabel="Test Accuracy (%)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot7_per_class_accuracy(results: dict, out: str) -> None:

    lams = sorted(results.keys())
    data = np.array([results[l]["per_class_acc"] for l in lams])
    row_labels = [f"λ={l:.0e}" for l in lams]

    fig, ax = plt.subplots(figsize=(12, max(3, len(lams) * 1.2 + 1)))
    fig.suptitle("Plot 7 — Per-Class Accuracy After Pruning (%)",
                 fontsize=14, fontweight="bold")

    im = ax.imshow(data, cmap="RdYlGn", aspect="auto",
                   vmin=max(0, data.min() - 5), vmax=100)
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
    ax.set_yticks(range(len(lams))); ax.set_yticklabels(row_labels)

    for i in range(len(lams)):
        for j in range(10):
            ax.text(j, i, f"{data[i, j]:.0f}",
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color="white" if data[i, j] < 55 else "black")

    plt.colorbar(im, ax=ax, label="Accuracy (%)", fraction=0.02, pad=0.01)
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def plot8_conv_gate_heatmap(results: dict, out: str) -> None:

    best_lam = min(results.keys(),
                   key=lambda l: abs(results[l]["sparsity"] - 0.5))
    model    = results[best_lam]["model"]
    gates    = model.conv1.get_gates()            # shape: (32, 3, 3, 3)

    # Rearrange for display: (32 out_ch × 3 in_ch, 3×3 kernels) side by side
    # Show first 16 output filters
    n_show = 16
    cols   = 8
    rows   = n_show // cols
    fig, axes = plt.subplots(rows * 3, cols,
                             figsize=(cols * 1.8, rows * 3 * 1.8))
    fig.suptitle(f"Plot 8 — conv1 Gate Heatmaps (λ={best_lam:.0e})\n"
                 f"Each 3×3 block = one spatial kernel gate pattern",
                 fontsize=13, fontweight="bold")

    for idx in range(n_show):
        for ch in range(3):    # RGB channels
            r   = (idx // cols) * 3 + ch
            c   = idx % cols
            ax  = axes[r, c]
            img = gates[idx, ch].numpy()        # 3×3
            ax.imshow(img, cmap="viridis", vmin=0, vmax=1, interpolation="nearest")
            if ch == 0: ax.set_title(f"F{idx}", fontsize=7, pad=2)
            ch_name = ["R","G","B"][ch]
            ax.set_ylabel(ch_name, fontsize=7, labelpad=1)
            ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")

In [ ]:
def run_experiment(base_lam: float,
                   train_loader: DataLoader,
                   test_loader:  DataLoader,
                   device: torch.device,
                   num_epochs:  int   = 30,
                   lr:          float = 3e-4,
                   temperature: float = 0.5,
                   patience:    int   = 8,
                   ckpt_path:   str   = "best.pt",
                   verbose:     bool  = True) -> dict:

    set_seed(SEED)
    model     = PrunableCNN(temperature=temperature).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history       = []
    best_acc      = -1.0
    no_improve    = 0

    for epoch in range(1, num_epochs + 1):
        lam_eff = lambda_schedule(base_lam, epoch, num_epochs)
        t0 = time.time()

        tl, cl, sl = train_one_epoch(
            model, train_loader, optimizer, criterion, lam_eff, device)
        acc, _ = evaluate(model, test_loader, device)
        spar   = model.overall_sparsity()
        lw_sp  = model.layerwise_sparsity()
        scheduler.step()

        history.append(dict(epoch=epoch, lam_eff=lam_eff,
                            total_loss=tl, cls_loss=cl, sp_loss=sl,
                            test_acc=acc, sparsity=spar,
                            lw_sparsity=lw_sp))

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            flag = "✓ new best"
        else:
            no_improve += 1
            flag = ""

        if verbose:
            lw_str = "  ".join(f"{n}:{v*100:.0f}%"
                                for n, v in lw_sp.items())
            print(f"  λ={base_lam:.0e} | ep {epoch:>2}/{num_epochs} | "
                  f"cls={cl:.3f} λ_eff={lam_eff:.2e} | "
                  f"acc={acc:.2f}%  spar={spar*100:.1f}%  [{lw_str}] "
                  f"| {time.time()-t0:.1f}s {flag}")

        if no_improve >= patience:
            print(f"  ⚡ Early stopping at epoch {epoch}")
            break

    # ── Load best checkpoint and evaluate ────────────────────────────────────
    model.load_state_dict(torch.load(ckpt_path, map_location=device,
                                     weights_only=True))
    acc_before, _ = evaluate(model, test_loader, device)

    pruned_model = hard_prune(model, threshold=1e-2)
    acc_after, per_class = evaluate(pruned_model, test_loader, device)

    delta = acc_before - acc_after
    print(f"\n  ▶ acc before hard prune: {acc_before:.2f}%")
    print(f"  ▶ acc after  hard prune: {acc_after:.2f}%  (Δ = {delta:+.2f}%)")
    print(f"  ▶ overall sparsity     : {model.overall_sparsity()*100:.1f}%")

    return dict(
        acc_before        = acc_before,
        acc_after         = acc_after,
        per_class_acc     = per_class,
        sparsity          = model.overall_sparsity(),
        final_lw_sparsity = model.layerwise_sparsity(),
        gate_values       = model.all_gate_values(),
        history           = history,
        model             = model,
        pruned_model      = pruned_model,
    )


In [ ]:
def main() -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs("outputs", exist_ok=True)

    print(f"\n{'═'*68}")
    print(f"  Self-Pruning CNN — CIFAR-10   │   device: {device}")
    print(f"{'═'*68}\n")

    train_loader, test_loader = get_cifar10_loaders(
        data_dir="./data", batch_size=128, num_workers=2)

    LAMBDA_VALUES = [1e-5, 1e-4, 1e-3]
    NUM_EPOCHS    = 35
    TEMPERATURE   = 0.5


    plot3_lambda_schedule(LAMBDA_VALUES, NUM_EPOCHS, "outputs/plot3_lambda_schedule.png")


    baseline = train_baseline(train_loader, test_loader, device, num_epochs=NUM_EPOCHS)


    results: Dict[float, dict] = {}
    for lam in LAMBDA_VALUES:
        print(f"\n{'─'*68}")
        print(f"  EXPERIMENT: λ_base = {lam:.0e}  |  "
              f"epochs = {NUM_EPOCHS}  |  τ = {TEMPERATURE}")
        print(f"{'─'*68}")
        results[lam] = run_experiment(
            base_lam     = lam,
            train_loader = train_loader,
            test_loader  = test_loader,
            device       = device,
            num_epochs   = NUM_EPOCHS,
            temperature  = TEMPERATURE,
            ckpt_path    = f"outputs/ckpt_lam{lam:.0e}.pt",
        )

    print(f"\n{'═'*68}")
    print("  FINAL RESULTS SUMMARY")
    print(f"{'═'*68}")
    print(f"  {'Lambda':<10} {'Acc (soft)':>12} {'Acc (hard)':>12} "
          f"{'Sparsity':>10}  {'vs Baseline':>12}")
    print(f"  {'─'*58}")
    for lam in LAMBDA_VALUES:
        r     = results[lam]
        delta = r["acc_after"] - baseline["best_acc"]
        print(f"  {lam:<10.0e} {r['acc_before']:>11.2f}% "
              f"{r['acc_after']:>11.2f}% "
              f"{r['sparsity']*100:>9.1f}%  "
              f"{delta:>+11.2f}%")
    print(f"  {'─'*58}")
    print(f"  {'Baseline':<10} {'—':>12} {baseline['best_acc']:>11.2f}%"
          f"{'0.0':>10}%  {'reference':>12}")
    print(f"{'═'*68}\n")


    print("Generating visualisations …")
    plot1_gate_distributions(results,               "outputs/plot1_gate_distributions.png")
    plot2_training_curves(results, baseline,        "outputs/plot2_training_curves.png")

    plot4_layerwise_sparsity(results,               "outputs/plot4_layerwise_sparsity.png")
    plot5_tradeoff(results, baseline["best_acc"],   "outputs/plot5_tradeoff.png")
    plot6_before_after(results,                     "outputs/plot6_before_after_pruning.png")
    plot7_per_class_accuracy(results,               "outputs/plot7_per_class_accuracy.png")
    plot8_conv_gate_heatmap(results,                "outputs/plot8_conv_gate_heatmap.png")
    print("\n✓ All outputs saved to outputs/\n")


if __name__ == "__main__":
    main()



════════════════════════════════════════════════════════════════════
  Self-Pruning CNN — CIFAR-10   │   device: cuda
════════════════════════════════════════════════════════════════════

Train samples: 50,000  |  Test samples: 10,000
Batch size: 128  |  Train batches: 391
  Saved → outputs/plot3_lambda_schedule.png

────────────────────────────────────────────────────────────
  Training BASELINE (no pruning) for fair comparison
────────────────────────────────────────────────────────────
  Epoch  1/35  |  Test acc: 57.52%
  Epoch  5/35  |  Test acc: 74.99%
  Epoch 10/35  |  Test acc: 80.98%
  Epoch 15/35  |  Test acc: 81.48%
  Epoch 20/35  |  Test acc: 84.74%
  Epoch 25/35  |  Test acc: 85.59%
  Epoch 30/35  |  Test acc: 86.84%
  Epoch 35/35  |  Test acc: 86.95%

  ▶ Baseline Best Accuracy: 87.03%


────────────────────────────────────────────────────────────────────
  EXPERIMENT: λ_base = 1e-05  |  epochs = 35  |  τ = 0.5
─────────────────────────────────────────────────────────────